# Label a set of responses with an encoder judge, then train

A detector is trained for **one model**: it scores how uncertain that model was while it
generated, so it is fitted on responses that model produced, plus a verdict on each. This
notebook starts from responses you already have and produces the missing half — the
verdicts — then fits a WEPR detector on the pair.

`artefactory/BERTJudge` grades an answer against a reference. Give it the question, the
answer to grade and the gold answer; it returns P(correct). It is a 210M encoder, efficient
enough to run on CPU: it downloads once (~420 MB), and every response after that is one
forward pass rather than an API request. Its own package, `bert-judge`, wraps the loading
and the scoring, so grading the whole file is one call; `THRESHOLD` turns the score into the 0/1 label
the detector is fitted on.

The sample files are synthetic — real questions, but the responses and their
log-probabilities were generated rather than sampled from a model. This notebook ships
without stored outputs because it downloads the judge's weights, so the numbers you see are
the ones your own run produces.

In [ ]:
# On Colab, uncomment to install the package and fetch the files this notebook reads.
# `bert-judge` is the judge's own package and declares no dependencies, so torch,
# transformers and datasets are named here. The transformers range is the checkpoint's:
# outside it the load fails, on `torch_dtype=` below and `KeyError: 'default'` at 5.x.
# !pip install -q artefactual torch 'transformers>=4.57,<5' datasets
# !pip install -q 'bert-judge @ git+https://github.com/artefactory/BERT-as-a-Judge.git'
# !wget -q https://raw.githubusercontent.com/artefactory/artefactual/main/docs/examples/responses_sample.jsonl
# !wget -q https://raw.githubusercontent.com/artefactory/artefactual/main/docs/examples/questions_sample.json

## The inputs

`RESPONSES` holds what the model under test generated, one OpenAI Batch line per request,
each carrying `top_logprobs` per token — that distribution is the only thing the detector
reads. `QUESTIONS` holds what each response is graded against: the question that was asked
and the gold answer, keyed by the `custom_id` the response carries.

`read_batch` reads the lines and `ChatCompletion` reads the text inside one — both the
library's, so the batch file is never indexed as a raw dict. A failed request carries no completion and
`failure` says why: two of the hundred lines are timeouts, which is what leaves 98
responses to judge.

In [ ]:
import json
from pathlib import Path

from artefactual.preprocessing import index_by_custom_id, read_batch
from artefactual.preprocessing.response_models import ChatCompletion

# Responses generated by the model this detector is being built for. This is the file to
# swap for your own run.
RESPONSES = Path("responses_sample.jsonl")
# What each of them is graded against: `question`, `short_answer`, and the `question_id`
# that is the response's `custom_id`.
QUESTIONS = Path("questions_sample.json")
# Where the judge's verdicts go, in the same Batch shape the responses arrive in.
JUDGMENTS = Path("judgments.jsonl")

# The checkpoint the model card recommends: trained on unconstrained generations, and
# reading question, candidate and reference.
JUDGE_MODEL = "artefactory/BERTJudge"
# P(correct) at or above which the response counts as correct.
THRESHOLD = 0.5
# Sequences per forward pass. Raise it on a GPU.
JUDGE_BATCH = 8
K = 15  # ranks per token; a detector is loaded at the k it was fit at
SEED = 42

lines = read_batch(RESPONSES)
for row in lines:
    if row.failure:
        print(f"dropped {row.custom_id}: {row.failure}")

generated = index_by_custom_id(lines)
questions = {entry["question_id"]: entry for entry in json.loads(QUESTIONS.read_text(encoding="utf-8"))}


def said(row):
    """What the model answered, read through the envelope rather than out of the raw dict."""
    return (ChatCompletion.model_validate(row.completion).choices[0].message.content or "").strip()


graded = [(questions[custom_id], row, said(row)) for custom_id, row in generated.items() if custom_id in questions]
print(f"{len(generated)} responses, {len(graded)} with a reference to grade against")

## Score the responses

`BERTJudge.predict` takes the three fields as parallel lists and returns one P(correct) per
response — the model card's quickstart, on this notebook's data.

The verdicts go to `judgments.jsonl` in the shape every other stage writes: a Batch line
per response, carrying `{"judgment": true|false, "score": ..., "explanation": ...}` as its
content. That is the pipeline's verdict format, not the API's, so `read_judgment`, the
{doc}`train_wepr` notebook and `scripts/train_detector.py` all read this file as they read a
generative judge's. The score stays beside the verdict, so the file says how close each call
was.

One caveat about the sample file: its responses are bare one-word strings, and this
checkpoint scores them erratically — some verbatim matches against the gold answer come back
below `THRESHOLD`. Point `RESPONSES` at sentence-shaped responses of your own before reading
anything into the numbers.

In [ ]:
from bert_judge.judges import BERTJudge

# float32 on CPU; pass "bfloat16" on a GPU, which is the checkpoint's own dtype.
judge = BERTJudge(model_path=JUDGE_MODEL, trust_remote_code=True, dtype="float32")

scores = judge.predict(
    questions=[question["question"] for question, _, _ in graded],
    candidates=[response for _, _, response in graded],
    references=[question["short_answer"] for question, _, _ in graded],
    batch_size=JUDGE_BATCH,
)

with JUDGMENTS.open("w", encoding="utf-8") as out:
    for (question, _, _), score in zip(graded, scores, strict=True):
        # `judgment` is the field every reader of this format looks at, and it says the
        # answer was CORRECT -- the opposite of the class the detector predicts. `score` and
        # `explanation` ride along so a verdict can be read back and argued with.
        verdict = {
            "judgment": bool(score >= THRESHOLD),
            "score": float(score),
            "explanation": f"BERTJudge P(correct)={score:.3f} against {question['short_answer']!r}",
        }
        out.write(
            json.dumps(
                {
                    "id": f"chatcmpl-judge-{question['question_id']}",
                    "custom_id": question["question_id"],
                    "response": {
                        "status_code": 200,
                        "body": {
                            "model": JUDGE_MODEL,
                            "choices": [
                                {
                                    "index": 0,
                                    "finish_reason": "stop",
                                    "message": {"role": "assistant", "content": json.dumps(verdict)},
                                }
                            ],
                        },
                    },
                    "error": None,
                },
                ensure_ascii=True,
            )
            + "\n"
        )

undecided = [score for score in scores if abs(score - THRESHOLD) < 0.1]
print(f"wrote {JUDGMENTS}, {len(scores)} scored")
print(f"  min {min(scores):.3f}, max {max(scores):.3f}, {len(undecided)} within 0.1 of THRESHOLD={THRESHOLD}")
for (question, _, response), score in list(zip(graded, scores, strict=True))[:3]:
    verdict = "hallucination" if score < THRESHOLD else "grounded    "
    print(f"  [{verdict}] {score:.3f}  said {response[:30]!r} (gold: {question['short_answer']!r})")

## Fit

This step reads both files back rather than using what is still in memory, which is what
makes it true that you can come back tomorrow, change `k`, and refit without scoring
anything again. `trainable=True` returns the pipeline unfitted, so `fit` takes the batch
lines directly and there is no feature extraction to write: `wepr`'s own parser opens the
Batch envelope and reads the `top_logprobs`. The split is stratified and happens first, so what is reported
describes responses the detector never saw.

Two numbers, and they answer different questions. **ROC-AUC** scores the ranking — whether
hallucinations sort above grounded responses — which is what matters if you triage by score.
The **classification report** scores the decisions at a 0.5 cut: recall on the
`hallucination` row is the fraction actually caught. Only the AUC carries over to a
different threshold.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from artefactual.preprocessing import read_judgment
from artefactual.scoring import wepr

# Read back from disk rather than from the variables above: this is the path a fresh kernel
# takes, and the one anything else reading these files takes too. `read_judgment` returns
# True when the judge said the response was CORRECT, so the label is its negation.
labelled = [
    (generated[row.custom_id], int(not verdict))
    for row in read_batch(JUDGMENTS)
    if row.custom_id in generated and (verdict := read_judgment(row.completion)) is not None
]

responses = [row for row, _ in labelled]
y = np.array([label for _, label in labelled])
print(f"read {len(responses)} labelled responses back from {RESPONSES.name} and {JUDGMENTS.name}")
print(f"{y.sum()} hallucinations ({y.mean():.0%})")

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=SEED)
detector = wepr(k=K, trainable=True).fit(x_train, y_train)
predicted = detector.predict_proba(x_test)[:, 1]

print(f"fitted on {len(y_train)}, holding out {len(y_test)}")
print(f"ROC-AUC: {roc_auc_score(y_test, predicted):.2f}\n")
print(classification_report(y_test, predicted >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

## Audit the labels the judge produced

An encoder judge gives a number and no explanation, so the audit has to come from
somewhere else. `cross_val_predict` gives every response a score from a fold that did not
contain it, and the responses the detector is *most* confident about while disagreeing with
the judge are where to look first: either the judge mislabelled that row, or the detector
found something real. Either way it is a handful of rows to read, not ninety-eight.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

folds = StratifiedKFold(5, shuffle=True, random_state=SEED)
# Out-of-fold: every score comes from a detector that never saw that response.
out_of_fold = cross_val_predict(wepr(k=K, trainable=True), responses, y, cv=folds, method="predict_proba")[:, 1]

scored = zip(out_of_fold, y, responses, strict=True)
disagreements = sorted(scored, key=lambda row: abs(row[0] - row[1]), reverse=True)[:5]

print("the five rows the detector disagrees with the judge about most:")
for score, label, row in disagreements:
    print(f"  judge said {'hallucination' if label else 'grounded    '}, detector P={score:.3f}  {said(row)[:40]!r}")

## Save it, and load it back

`.skops` rather than a pickle. The same call loads it back from three kinds of identifier:
a `.skops` file, a directory holding `model.skops`, or **any Hugging Face repository id** —
in which case `model.skops` is downloaded from that repo and cached. That is why publishing
one means naming the file `model.skops`.

`k` is part of the weights, not a runtime option: the coefficients were fitted at one rank
count and mean nothing at another, so loading at a different `k` raises rather than
mis-shaping the score.

In [ ]:
path = detector.save_estimator("wepr-bertjudge.skops")
reloaded = wepr(path, k=K)

# Held-out responses: the rows the fit above never saw.
for row, label in list(zip(x_test, y_test, strict=True))[:5]:
    print(
        f"[{'hallucination' if label else 'grounded    '}] P={reloaded.predict_proba(row)[0, 1]:.3f}  {said(row)[:40]!r}"
    )

reloaded

## Where to go next

- **Choose the threshold rather than accept it.** `judgments.jsonl` keeps the raw score
  beside the verdict, so moving `THRESHOLD` and re-running the scoring cell relabels the run
  in seconds — the download is already paid for. If the distribution printed above had a
  pile near the cut, this is the knob that decided those labels.
- **Compare against a generative judge.** Ask one for a verdict on the same responses and
  join the two files on `custom_id`. Where they disagree is where the alias list mattered —
  this judge reads `short_answer` alone — or where one of them is wrong.
- **Feed the CLI.** `judgments.jsonl` and `responses_sample.jsonl` go straight into the
  `train_detector.py` script in
  [the repository](https://github.com/artefactory/artefactual/blob/main/scripts/train_detector.py),
  which also reports the bootstrap confidence intervals a holdout this size needs.

- **Another judge checkpoint.** `artefactory/BERTJudge-Free-CR` drops the question from the
  input, and the `Formatted` family expects responses that end in `Final answer: <x>`. The
  paper's own recommendation is the one set above.
- **Your own responses.** Only section 1 changes: point `RESPONSES` at a Batch output file
  from the model you want to detect hallucinations in, and `QUESTIONS` at what to grade it
  against.